In [7]:
import sys
from pathlib import Path

src_path = Path(
    "/home/xls/workspace/projects/qwen3-tn-compression/src"
).resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("使用源码路径：", src_path)

使用源码路径： /mnt/intern7/xls/projects/qwen3-tn-compression/src


In [8]:
import torch
from torch.nn import functional as F

from qwen3_tn import (
    TTMatrixLinear,
    TTMatrixSpec,
    detensorize_matrix,
    reconstruct_matrix,
    tensorize_matrix,
    tt_svd_matrix,
)

创建矩阵

In [9]:
torch.manual_seed(7)

weight = torch.randn(8, 8, dtype=torch.float64)

spec = TTMatrixSpec.full_rank(
    out_modes=(2, 2, 2),
    in_modes=(2, 2, 2),
)

# 设置两个内部 bond dimension
# spec = spec.with_bond_rank(bond_index=1, rank=2)
# spec = spec.with_bond_rank(bond_index=2, rank=3)

print("矩阵形状：", weight.shape)
print("TT ranks：", spec.ranks)

矩阵形状： torch.Size([8, 8])
TT ranks： (1, 4, 4, 1)


验证矩阵张量化

In [10]:
tensorized = tensorize_matrix(weight, spec)
restored = detensorize_matrix(tensorized, spec)

print("张量化后的形状：", tensorized.shape)
print("能否精确恢复：", torch.equal(weight, restored))

assert torch.equal(weight, restored)

张量化后的形状： torch.Size([4, 4, 4])
能否精确恢复： True


验证 TT-matrix SVD

In [11]:
cores = tt_svd_matrix(
    weight,
    spec,
    svd_driver=None,
)

for index, core in enumerate(cores):
    print(f"core {index}：{tuple(core.shape)}")

reconstructed = reconstruct_matrix(cores, spec)

relative_error = (
    torch.linalg.vector_norm(reconstructed - weight)
    / torch.linalg.vector_norm(weight)
)

print("相对重构误差：", relative_error.item())

# assert relative_error.item() < 1e-10

core 0：(1, 2, 2, 4)
core 1：(4, 2, 2, 4)
core 2：(4, 2, 2, 1)
相对重构误差： 1.6406533339167614e-15


验证 TTMatrixLinear forward

In [12]:
layer = TTMatrixLinear(
    spec,
    cores,
    trainable=False,
    preserve_input_dtype=False,
)

inputs = torch.randn(5, 8, dtype=torch.float64)

dense_output = F.linear(inputs, weight)
tt_output = layer(inputs)

torch.testing.assert_close(
    tt_output,
    dense_output,
    rtol=1e-10,
    atol=1e-10,
)

print("TTMatrixLinear forward 验证通过")

TTMatrixLinear forward 验证通过
